# Coffee Roasting Implementation

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Normalization

### 1. Generate 200 random points for Temp (150 to 280°C) and Duration (10 to 18 mins)


In [ ]:
def generate_coffee_data():
    np.random.seed(42)
    X = np.random.rand(200,2)
    X[:,0] = X[:,0] * (280-150) + 150 # Temp
    X[:,1] = X[:,1] * (18-10) + 10 # Duration
    Y = np.zeros((200, 1))
    
    # Define the "good roast" decision boundaries (recreating the original triangle/polygon)
    for i in range(len(X)):
        t, d = X[i,0], X[i,1]
        # criteria for a "Good Roast" (Y=1)
        if (12 <= d <= 15) and (175 <= t <= 260):
            # As temp goes up, duration needs to be on the lower end
            if (t - 175) / (260 - 175) + (d - 12) / (15 - 12) <= 1.5:
                Y[i] = 1
                
    return X, Y
    

In [ ]:
X, Y = generate_coffee_data()
print(f"Data Shapes -> X: {X.shape}, Y: {Y.shape}")

### 2 Feature Normalization

In [ ]:
norm_layer = Normalization(axis = -1)
norm_layer.adapt(X)  # Learns the mean and variance of our data
Xn = norm_layer(X) # normalizing data

### 3.  Tiling Data
The lab copies the 200 samples 1,000 times to create a larger training loop dataset.

In [ ]:
Xt = np.tile(Xn, (1000, 1))
Yt = np.tile(Y, (1000, 1))


### 4. TensorFlow Model

In [ ]:
tf.random.set_seed(1234)
model = Sequential([
    tf.keras.Input(shape = (2,)),
    Dense(units=3, activation='sigmoid', name = 'layer1'),
    Dense(units=1, activation='linear', name = 'layer2')
])

model.summary()

### 5. Compile and Train Model

In [ ]:
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
)

print("\nTraining the network...")

model.fit(Xt, Yt, epochs=10, batch_size=32, verbose=1)

### 6. MAKE PREDICTIONS (INFERENCE)

In [ ]:
X_test = np.array([
    [200.0, 13.9],
    [200.0, 1.9]
])

# Imp step: Normalize test data exactly like training data!
X_testn = norm_layer(X_test)

logits = model.predict(X_testn)
predictions = tf.nn.sigmoid(logits).numpy()
print("\nPredictions (Probabilities):")

for i, pred in enumerate(predictions):
    print(f"Target {X_test[i]} -> Probability of Good Roast: {pred[0]:.4f} -> Decision: {'Good Roast! ☕' if pred[0] >= 0.5 else 'Bad Roast 🤢'}")